In [ ]:
import pandas as pd

path = "../data/raw/2023_PM10_1g.xlsx"
df = pd.read_excel(path)

df.set_index('DateTime', inplace=True)

target_column = df.columns[1]

target_column, df[target_column].isna().sum()

('DsDziePilsud', 191)

In [ ]:
# Re-import necessary libraries
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Drop rows where the target column is NaN for training
df_train = df.dropna(subset=[target_column])

# Define Features (X) and Target (y)
X = df_train.drop(columns=[target_column])  # All columns except target
y = df_train[target_column]  # Target column

# Standardize features
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

# Standardize target variable
scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1)).flatten()

# Split into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

# Define MLP model
model = keras.Sequential([
    keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1)  # Output layer
])

# Compile model
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Train model
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_val, y_val), verbose=0)

# Prepare data for imputing missing values
df_missing = df[df[target_column].isna()].drop(columns=[target_column])  # Features only
X_missing_scaled = scaler_X.transform(df_missing)  # Scale missing data

# Predict missing values
y_missing_pred_scaled = model.predict(X_missing_scaled)
y_missing_pred = scaler_y.inverse_transform(y_missing_pred_scaled).flatten()  # Convert back to original scale

# Fill missing values in the DataFrame
df.loc[df[target_column].isna(), target_column] = y_missing_pred

C:\Users\Jakub\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


ModuleNotFoundError: No module named 'ace_tools'

In [10]:
history.params

{'verbose': 0, 'epochs': 50, 'steps': 215}